# Bitcoin Market Sentiment × Trader Performance Analysis
**Primetrade.ai — Hyperliquid Historical Data**

Full pipeline: EDA → Statistics → ML

In [ ]:
import sys, os
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
get_ipython().run_line_magic("matplotlib", "inline")
print("Libraries loaded OK")

## 1. Load & Preprocess

In [ ]:
from src.preprocessing import load_and_prepare
os.chdir("..")  # run from project root
df = load_and_prepare("data/fear_greed.csv", "data/historical_data.csv")
df.head(3)

In [ ]:
df[["closedPnL","position_value","sentiment_encoded","win"]].describe().round(4)

## 2. Exploratory Data Analysis

In [ ]:
from src.eda import *
OUT = "outputs/figures"
os.makedirs(OUT, exist_ok=True)

In [ ]:
plot_sentiment_distribution(df, OUT); plt.show()

In [ ]:
plot_daily_profit_trend(df, OUT); plt.show()

In [ ]:
plot_pnl_by_sentiment_violin(df, OUT); plt.show()

In [ ]:
plot_avg_pnl_by_sentiment(df, OUT); plt.show()

In [ ]:
plot_win_rate_by_sentiment(df, OUT); plt.show()

In [ ]:
plot_position_size_by_sentiment(df, OUT); plt.show()

In [ ]:
plot_direction_by_sentiment(df, OUT); plt.show()

In [ ]:
plot_long_vs_short_pnl(df, OUT); plt.show()

In [ ]:
plot_correlation_heatmap(df, OUT); plt.show()

In [ ]:
plot_cumulative_pnl(df, OUT); plt.show()

## 3. Statistical Tests

In [ ]:
stats_results = run_statistical_tests(df)
print_statistical_summary(stats_results)

## 4. Trader Segmentation

In [ ]:
seg_df = trader_segmentation(df)
print_segment_summary(seg_df)
seg_df.head(10)

## 5. Machine Learning

In [ ]:
from src.modeling import train_all_models
ml_results = train_all_models(df, output_dir_figs="outputs/figures", output_dir_models="outputs/models")

## 6. Final Insights

In [ ]:
avg_pnl = df.groupby("Classification")["closedPnL"].mean().sort_values(ascending=False)
win_rate = df.groupby("Classification")["win"].mean().mul(100).round(2)
insights = pd.DataFrame({"Avg PnL (USD)": avg_pnl.round(4), "Win Rate (%)": win_rate})
print("=== Sentiment Performance Summary ===")
print(insights.to_string())
best = avg_pnl.idxmax()
print(f"\nMost profitable sentiment: {best} (avg PnL = ${avg_pnl[best]:.4f})")
if "ttest" in stats_results:
    t = stats_results["ttest"]
    print(f"T-Test p={t['p_value']:.6f} ({'significant' if t['significant'] else 'not significant'})")
best_ml = max([(n,r) for n,r in ml_results.items() if r["roc_auc"]], key=lambda x: x[1]["roc_auc"])
print(f"Best ML model: {best_ml[0]} (ROC-AUC={best_ml[1]['roc_auc']:.4f})")